In [1]:
import os
import json
import pandas as pd
import time
import random
from datetime import datetime, timedelta
import faiss
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer

c:\Users\Danh\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def _get_store_paths(store_name: str):
    """Tạo đường dẫn file động cho một kho tri thức cụ thể."""
    base_dir = r"D:\finalproject\KLTN\Backend\data\vector_store"
    index_path = os.path.join(base_dir, f"faiss_index_{store_name}.bin")
    docs_path = os.path.join(base_dir, f"documents_{store_name}.json")
    return index_path, docs_path

_stores = {}

def get_store(store_name: str):
    """
    Lấy một kho tri thức cụ thể. Tải từ cache nếu có, nếu không thì xây dựng mới.
    """
    if store_name in _stores:
        return _stores[store_name]

    index_path, docs_path = _get_store_paths(store_name)

    if os.path.exists(index_path) and os.path.exists(docs_path):
        try:
            print(f"Đang tải kho '{store_name}' từ cache...")
            index = faiss.read_index(index_path)
            with open(docs_path, 'r', encoding='utf-8') as f:
                documents = json.load(f)
            print(f"Tải thành công kho '{store_name}' với {index.ntotal} vector.")
            
            store_instance = {"index": index, "documents": documents}
            _stores[store_name] = store_instance
            return store_instance
        except Exception as e:
            print(f"Lỗi khi tải kho '{store_name}' từ cache: {e}. Sẽ xây dựng lại.")

def retrieve(store_name: str, query: str, k: int = 5) -> str:
    """Thực hiện truy vấn trên một kho tri thức chuyên biệt."""
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
    store = get_store(store_name)
    if not store or store.get("index") is None:
        print(f"Truy vấn thất bại: Kho tri thức '{store_name}' chưa được khởi tạo.")
        return f"Lỗi: Cơ sở tri thức '{store_name}' không khả dụng."

    index = store["index"]
    documents = store["documents"]
    
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )
    
    try:
        _, indices = index.search(np.array(query_embedding, dtype=np.float32), k)
        retrieved_docs = [documents[i] for i in indices[0]]
        context = "\n---\n".join([doc['content'] for doc in retrieved_docs])
        
        print(f"Đã truy xuất {len(retrieved_docs)} đoạn văn bản từ kho '{store_name}' cho câu hỏi: '{query[:50]}...'")
        return context
    except Exception as e:
        print(f"Lỗi trong quá trình truy xuất từ kho '{store_name}': {e}")
        return "Lỗi: Đã xảy ra sự cố khi tìm kiếm thông tin."

In [6]:
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

In [59]:
import json
from typing import Dict # Thêm type hints cho Dict
# Giả định retrieve là hàm đã được định nghĩa và days đã được tính chính xác

import json
from typing import Dict 
# Giả định retrieve là hàm đã được định nghĩa

def _dynamic_retrieval(detected_disease: str, iot_data: Dict) -> str:
    """
    Thực hiện truy xuất kiến thức Cấp 1 (Sơ bộ) cho LLM Orchestrator.
    Sử dụng ngày tuổi (days) và thông tin thời tiết đã có sẵn trong iot_data.
    """
    all_context = []
    
    days = iot_data.get('days_after_planting', 50) 
    temp = iot_data.get('temperature', 28)
    humidity = iot_data.get('humidity', 75)
    soil_moisture = iot_data.get('soil_moisture', 60)
    
    if temp > 30 and humidity < 70:
        weather_text = "nắng nóng và khô hanh"
    elif temp < 25 and humidity > 90:
        weather_text = "mát mẻ và ẩm ướt (có nguy cơ bệnh)"
    else:
        weather_text = "ổn định"
    
    if detected_disease and detected_disease != 'healthy':
        query = f"Nguyên tắc ưu tiên điều trị bệnh {detected_disease} cho lúa giai đoạn {days} NSS"
        disease_context = retrieve(detected_disease, query, k=5) 
        all_context.append(f"--- KIẾN THỨC BỆNH/SÂU HẠI ({detected_disease.upper()}) ---\n{disease_context}")
    is_water_critical = soil_moisture < 30 
    
    if is_water_critical:
        query = f"Nguyên tắc quản lý nước khẩn cấp (tưới) cho lúa {days} ngày tuổi trong điều kiện {weather_text} và độ ẩm đất {soil_moisture}%"
    else:
        query = f"Nguyên tắc bón phân và duy trì mực nước cho lúa {days} ngày tuổi trong điều kiện thời tiết {weather_text}"
    
    general_context = retrieve('general_qa', query, k=5)
    all_context.append(f"--- KIẾN THỨC QUẢN LÝ CHUNG ---\n{general_context}")

    return "\n\n".join(all_context)

def _build_orchestration_prompt(detected_disease: str, iot_data: dict, retrieved_context: str) -> str:
    """Xây dựng prompt chuyên sâu cho việc điều phối."""

    prompt = f"""
        **Bối cảnh:** Bạn là Trưởng Agent Điều phối cho hệ thống nông nghiệp thông minh.
        
        **MỤC TIÊU:** Phân tích tất cả dữ liệu dưới đây để quyết định **CHUỖI HÀNH ĐỘNG TỐI ƯU** cho nông hộ trong 3 ngày tới, đảm bảo ưu tiên xử lý rủi ro cao nhất (Bệnh > Nước > Dinh Dưỡng).

        **DỮ LIỆU ĐẦU VÀO:**
        1. **PHÂN TÍCH ẢNH (Bệnh):** {detected_disease}
        2. **DỮ LIỆU BỐI CẢNH (IoT, Thời tiết, Giai đoạn):**
        ```json
        {iot_data}
        ```
        3. **KIẾN THỨC NỀN (Truy xuất từ Vector Store):**
        ```
        {retrieved_context}
        ```

        **QUY TẮC QUYẾT ĐỊNH (PHẢI LÀM):**
        A. **Ưu tiên 1 (Bệnh):** Nếu {detected_disease} khác 'healthy', BẮT BUỘC có `treatment_agent` trong chuỗi hành động.
        B. **Ưu tiên 2 (Nước/Thời tiết):**
            - Nếu `risk_of_heavy_rain` là true HOẶC `soil_moisture_percent` < 30, BẮT BUỘC `water_agent` phải có `priority` 1 hoặc 2.
            - Trong trường hợp Mưa Lớn, phải ưu tiên WaterAgent (Tháo nước/Chuẩn bị) hơn NutrientAgent.
        C. **Ưu tiên 3 (Dinh dưỡng):** Chỉ kích hoạt `nutrient_agent` khi điều kiện thời tiết ổn định và không có rủi ro nước hoặc bệnh nghiêm trọng.
        
        **YÊU CẦU ĐẦU RA (JSON BẮT BUỘC):**
        - `reasoning` PHẢI lồng ghép các số liệu cụ thể (ví dụ: "Độ ẩm đất là 75%, nhưng dự báo có mưa lớn, nên ưu tiên WaterAgent để tháo nước trước khi bón phân").
        - `action_sequence` phải SẮP XẾP theo thứ tự ưu tiên tăng dần (1 là cao nhất).
        
        ```json
        {{
            "orchestration_decision": "True/False" (boolean),
            "main_focus": "string (TREATMENT/WATER/NUTRIENT/MONITORING)",
            "reasoning": "string (Giải thích ưu tiên dựa trên số liệu)",
            "action_sequence": [
                {{
                    "agent": "water_agent",
                    "priority": 1,
                    "purpose": "Tháo nước khẩn cấp do mưa lớn"
                }},
                // ... các hành động tiếp theo
            ]
        }}
        ```
    """
    return prompt

def _execute_llm_decision(decision: dict, iot_data: dict, analysis_result: dict, context_data):
    """[Giả lập] Chỉ ghi lại quyết định, không gọi agent thực thi."""
    execution_log = []

    if not decision.get("orchestration_decision"):
        log_msg = f"LLM quyết định không cần hành động ưu tiên. Lý do: {decision.get('reasoning')}"
        print(f"[ORCHESTRATOR] {log_msg}")
        execution_log.append(log_msg)
        return execution_log

    actions = sorted(decision.get('action_sequence', []), key=lambda x: x.get('priority', 99))

    for action in actions:
        agent_name = action.get('agent')
        purpose = action.get('purpose', 'Không xác định')
        log_msg = f"[SIMULATION] {agent_name} sẽ được kích hoạt với mục đích: {purpose}"
        print(log_msg)
        execution_log.append(log_msg)

    return execution_log
    
def check_risk_for_farmer_for_evaluation(iot_data=None):
    """
    [ORCHESTRATOR] Chức năng điều phối chính, ĐÃ ĐIỀU CHỈNH cho mục đích ĐÁNH GIÁ:
    Bỏ qua phân tích ảnh, nhận tình trạng bệnh trực tiếp từ iot_data.
    """
    disease_map = {
        "bacterial_leaf_blight": "Cháy bìa lá", "blast": "Đạo ôn",
        "brown_spot": "Đốm nâu", "healthy": "Khỏe mạnh"
    }

    detected_disease = iot_data.get('detected_disease_name', 'healthy') 
    
    if detected_disease not in disease_map:
        detected_disease = 'healthy'
        
    disease_name_vn = disease_map.get(detected_disease, "Khỏe mạnh")
    
    analysis_data = {
        "detected_disease_name": detected_disease,
        "image_path_to_save": None
    }
    
    retrieved_context = _dynamic_retrieval(detected_disease, iot_data)
    
    try:
        prompt = _build_orchestration_prompt(disease_name_vn, iot_data, retrieved_context)
        

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}, 
            temperature=0.1
        )
        decision_json = json.loads(response.choices[0].message.content)

        execution_log = _execute_llm_decision(decision_json, iot_data, analysis_data, iot_data)
        
        return {
            "decision": decision_json,
            "execution_log": execution_log,
            "status": "orchestration_complete"
        }

    except Exception as e:
        print(f"[ORCHESTRATOR] Lỗi trong quá trình ra quyết định/thực thi bằng LLM: {e}")
        return {"error": "Lỗi hệ thống khi ra quyết định điều phối."}

In [60]:
def load_test_scenarios():
    return [
        # ===============================================================
        # 1-6: KỊCH BẢN ĐƠN LẺ – Chỉ 1 vấn đề nghiêm trọng
        # ===============================================================
        {
            "name": "S1: Đạo ôn lá nặng – giai đoạn đẻ nhánh",
            "iot_data": {
                "days_after_planting": 45,
                "temperature": 28.5, "humidity": 88, "soil_moisture": 68, "soil_ph": 5.8, "water_level": 12,
                "detected_disease_name": "blast", "disease_confidence": 0.92
            },
            "expected_main_focus": "TREATMENT",
            "expected_priority": [{"agent": "treatment_agent", "priority": 1, "purpose": "Phun khẩn Tricyclazole + Kasugamycin"}]
        },
        {
            "name": "S2: Đốm nâu nặng – giai đoạn làm đòng",
            "iot_data": {
                "days_after_planting": 62,
                "temperature": 27.0, "humidity": 92, "soil_moisture": 78, "soil_ph": 5.6, "water_level": 18,
                "detected_disease_name": "brown_spot", "disease_confidence": 0.89
            },
            "expected_main_focus": "TREATMENT",
            "expected_priority": [{"agent": "treatment_agent", "priority": 1}]
        },
        {
            "name": "S3: Cháy bìa lá do nấm + thiếu Kali nghiêm trọng",
            "iot_data": {
                "days_after_planting": 78,
                "temperature": 31.0, "humidity": 72, "soil_moisture": 58, "leaf_analysis_k": 0.85,
                "detected_disease_name": "brown_spot", "disease_confidence": 0.87
            },
            "expected_main_focus": "TREATMENT",
            "expected_priority": [{"agent": "treatment_agent", "priority": 1}]
        },
        {
            "name": "S4: Đạo ôn cổ bông – giai đoạn trổ đều",
            "iot_data": {
                "days_after_planting": 80,
                "temperature": 29.5, "humidity": 85, "soil_moisture": 65, "water_level": 8,
                "detected_disease_name": "neck_blast", "disease_confidence": 0.94
            },
            "expected_main_focus": "TREATMENT",
            "expected_priority": [{"agent": "treatment_agent", "priority": 1, "purpose": "Phun khẩn Tricyclazole 2 lần cách 5 ngày"}]
        },
        {
            "name": "S5: Thiếu nước nặng – giai đoạn trổ bông",
            "iot_data": {
                "days_after_planting": 82,
                "temperature": 33.0, "humidity": 48, "soil_moisture": 18, "water_level": 2,
                "detected_disease_name": "healthy"
            },
            "expected_main_focus": "WATER",
            "expected_priority": [{"agent": "water_agent", "priority": 1, "purpose": "Tưới khẩn cấp giữ nước mặt 5–8cm"}]
        },
        {
            "name": "S6: Ngập úng + dự báo mưa lớn",
            "iot_data": {
                "days_after_planting": 70,
                "temperature": 26.5, "humidity": 95, "soil_moisture": 100, "water_level": 42,
                "risk_of_heavy_rain": True,
                "detected_disease_name": "healthy"
            },
            "expected_main_focus": "WATER",
            "expected_priority": [{"agent": "water_agent", "priority": 1, "purpose": "Tháo nước khẩn cấp chuẩn bị mưa lớn"}]
        },

        # ===============================================================
        # 7-14: XUNG ĐỘT ƯU TIÊN – Bệnh > Nước > Dinh dưỡng
        # ===============================================================
        {
            "name": "C1: Đạo ôn lá + Thiếu nước nặng",
            "iot_data": {
                "days_after_planting": 58,
                "temperature": 30.0, "humidity": 82, "soil_moisture": 22, "water_level": 4,
                "detected_disease_name": "blast", "disease_confidence": 0.90
            },
            "expected_priority": [
                {"agent": "treatment_agent", "priority": 1},
                {"agent": "water_agent", "priority": 2, "purpose": "Tưới sau khi thuốc khô"}
            ]
        },
        {
            "name": "C2: Đốm nâu + Ngập úng",
            "iot_data": {
                "days_after_planting": 75,
                "temperature": 26.0, "humidity": 94, "soil_moisture": 100, "water_level": 38,
                "detected_disease_name": "brown_spot", "disease_confidence": 0.88
            },
            "expected_priority": [
                {"agent": "treatment_agent", "priority": 1},
                {"agent": "water_agent", "priority": 2}
            ]
        },
        {
            "name": "C3: Đạo ôn cổ bông + Mưa lớn sắp tới",
            "iot_data": {
                "days_after_planting": 81,
                "temperature": 28.0, "humidity": 90, "risk_of_heavy_rain": True,
                "detected_disease_name": "neck_blast", "disease_confidence": 0.93
            },
            "expected_priority": [{"agent": "treatment_agent", "priority": 1, "purpose": "Phun ngay trước mưa"}]
        },
        {
            "name": "C4: Đốm nâu + Thiếu Kali nghiêm trọng",
            "iot_data": {
                "days_after_planting": 68,
                "temperature": 30.5, "humidity": 78, "leaf_analysis_k": 0.88,
                "detected_disease_name": "brown_spot", "disease_confidence": 0.86
            },
            "expected_priority": [
                {"agent": "treatment_agent", "priority": 1},
                {"agent": "nutrient_agent", "priority": 2}
            ]
        },
        {
            "name": "C5: Đạo ôn + Hạn nặng + Thiếu Kali (3 vấn đề)",
            "iot_data": {
                "days_after_planting": 65,
                "temperature": 32.0, "humidity": 62, "soil_moisture": 16, "water_level": 3,
                "leaf_analysis_k": 0.90,
                "detected_disease_name": "blast", "disease_confidence": 0.91
            },
            "expected_priority": [
                {"agent": "treatment_agent", "priority": 1},
                {"agent": "water_agent", "priority": 2},
                {"agent": "nutrient_agent", "priority": 3}
            ]
        },

        # ===============================================================
        # 15-20: KHÔNG CẦN HÀNH ĐỘNG & CẬN BIÊN
        # ===============================================================
        {
            "name": "E1: Ruộng hoàn toàn khỏe – giai đoạn chắc xanh",
            "iot_data": {
                "days_after_planting": 92,
                "temperature": 29.0, "humidity": 78, "soil_moisture": 62, "soil_ph": 5.9, "water_level": 10,
                "detected_disease_name": "healthy"
            },
            "expected_orchestration_decision": False
        },
        {
            "name": "E2: Đã thu hoạch (>105 ngày)",
            "iot_data": {
                "days_after_planting": 108,
                "soil_moisture": 40,
                "detected_disease_name": "healthy"
            },
            "expected_orchestration_decision": False
        },
        {
            "name": "B1: Phát hiện đốm nâu nhưng confidence thấp",
            "iot_data": {
                "days_after_planting": 60,
                "detected_disease_name": "brown_spot",
                "disease_confidence": 0.42
            },
            "expected_orchestration_decision": False
        },
        {
            "name": "B2: Độ ẩm đất 29% → Khẩn cấp tưới",
            "iot_data": {
                "days_after_planting": 85,
                "soil_moisture": 29,
                "detected_disease_name": "healthy"
            },
            "expected_priority": [{"agent": "water_agent", "priority": 1}]
        },
        {
            "name": "B3: Độ ẩm đất đúng 30% → An toàn",
            "iot_data": {
                "days_after_planting": 70,
                "soil_moisture": 30,
                "detected_disease_name": "healthy"
            },
            "expected_orchestration_decision": False
        },
        {
            "name": "C6: Mưa lớn sắp tới + Đạo ôn nặng",
            "iot_data": {
                "days_after_planting": 55,
                "risk_of_heavy_rain": True,
                "detected_disease_name": "blast",
                "disease_confidence": 0.91
            },
            "expected_priority": [{"agent": "treatment_agent", "priority": 1, "purpose": "Phun ngay trước mưa"}]
        },
        {
            "name": "C7: Đốm nâu + Thiếu nước + Giai đoạn đón đòng",
            "iot_data": {
                "days_after_planting": 64,
                "temperature": 31.0, "humidity": 68, "soil_moisture": 24,
                "detected_disease_name": "brown_spot",
                "disease_confidence": 0.88
            },
            "expected_priority": [
                {"agent": "treatment_agent", "priority": 1},
                {"agent": "water_agent", "priority": 2},
                {"agent": "nutrient_agent", "priority": 3}
            ]
        }
    ]

In [61]:
import unidecode
from unittest.mock import MagicMock

# --- CLASSES GIẢ LẬP ---
class MockFarm:
    def __init__(self, province: str = "Hau Giang", area_ha: float = 2.0, planting_date: datetime.date = datetime.now().date(), rice_variety: str = 'om5451'):
        self.province = province
        self.area_ha = area_ha
        self.planting_date = planting_date
        self.rice_variety = rice_variety
        
class MockUser:
    def __init__(self, user_id: str, farm: MockFarm):
        self.id = user_id
        self.farms = [farm]

# --- HÀM SO SÁNH CHUỖI HÀNH ĐỘNG ---
def compare_action_sequences(actual, expected):
    # Ép priority về số
    for item in actual:
        pr = item.get("priority")
        if isinstance(pr, str):
            try:
                item["priority"] = int(pr)
            except:
                item["priority"] = 99

    for item in expected:
        pr = item.get("priority")
        if isinstance(pr, str):
            try:
                item["priority"] = int(pr)
            except:
                item["priority"] = 99

    # Sort lại
    sorted_actual = sorted(actual, key=lambda x: x.get('priority', 99))
    sorted_expected = sorted(expected, key=lambda x: x.get('priority', 99))

    # Soft matching: expected phải nằm trong actual
    for exp in sorted_expected:
        found_match = False
        for act in sorted_actual:
            if act.get("agent") == exp.get("agent") and act.get("priority") == exp.get("priority"):
                found_match = True
                break
        if not found_match:
            return False

    return True


In [62]:
def evaluate_orchestration_agent(client: MagicMock):
    """
    Thực hiện kiểm thử 44 kịch bản, đánh giá logic điều phối và tổng hợp kết quả.
    """
    scenarios = load_test_scenarios()
    results = []
    
    total_tests = len(scenarios)
    
    # Các biến đếm cho chỉ số chính
    correct_main_focus_count = 0
    correct_priority_1_count = 0
    correct_full_sequence_count = 0
    correct_decision_count = 0 # Đúng về orchestration_decision (True/False)
    
    print(f"--- BẮT ĐẦU ĐÁNH GIÁ AGENT ĐIỀU PHỐI ({total_tests} KỊCH BẢN) ---")

    for i, scenario in enumerate(scenarios):
        
        # 2. Thực thi Agent
        result = check_risk_for_farmer_for_evaluation(scenario['iot_data'])
        
        # 3. Phân tích Kết quả Trả về
        decision = result.get('decision', {})
        
        actual_main_focus = decision.get("main_focus")
        actual_decision_status = decision.get("orchestration_decision")
        
        actual_action_sequence = decision.get('action_sequence', [])
        
        expected_main_focus = scenario.get('expected_main_focus')
        expected_priority_sequence = scenario.get('expected_priority', [])
        expected_decision_status = scenario.get('expected_orchestration_decision', True) # Mặc định là True nếu không định nghĩa

        # 4. Tính toán Chỉ số
        
        # A. Độ chính xác Quyết định (True/False)
        is_decision_correct = (actual_decision_status == expected_decision_status)
        if is_decision_correct:
            correct_decision_count += 1
            
        # B. Độ chính xác Trọng tâm (Chỉ áp dụng khi quyết định là True)
        is_main_focus_correct = (actual_main_focus == expected_main_focus) if expected_main_focus else True
        if is_main_focus_correct:
             correct_main_focus_count += 1

        # C. Độ chính xác Ưu tiên 1
        expected_p1_agent = expected_priority_sequence[0].get('agent') if expected_priority_sequence else None
        actual_p1_agent = None
        if actual_action_sequence:
            # Ép priority
            for item in actual_action_sequence:
                if isinstance(item.get("priority"), str):
                    try:
                        item["priority"] = int(item["priority"])
                    except:
                        item["priority"] = 99

            p1 = sorted(actual_action_sequence, key=lambda x: x.get("priority", 99))[0]
            actual_p1_agent = p1.get("agent")
        
        is_priority_1_correct = (actual_p1_agent == expected_p1_agent)
        if is_priority_1_correct:
            correct_priority_1_count += 1
            
        # D. Độ chính xác Toàn bộ Chuỗi
        is_full_sequence_correct = compare_action_sequences(actual_action_sequence, expected_priority_sequence)
        if is_full_sequence_correct:
            correct_full_sequence_count += 1

        results.append({
            "Scenario": scenario['name'],
            "Detected_Disease": scenario['iot_data'].get('detected_disease_name') or 'healthy',
            "Expected_Decision": expected_decision_status,
            "Actual_Decision": actual_decision_status,
            "Decision_Match": is_decision_correct,
            "Expected_Focus": expected_main_focus or 'N/A',
            "Actual_Focus": actual_main_focus,
            "Focus_Match": is_main_focus_correct,
            "P1_Agent_Match": is_priority_1_correct,
            "Full_Sequence_Match": is_full_sequence_correct,
            "LLM_Reasoning": decision.get('reasoning', 'N/A')
        })
        
        print(f"[{i+1}/{total_tests}] {scenario['name']} - P1 Match: {'✅' if is_priority_1_correct else '❌'}")
    
    # 5. Tổng hợp Kết quả
    summary = {
        "Total_Scenarios": total_tests,
        "Decision_Accuracy": f"{(correct_decision_count / total_tests) * 100:.2f}% ({correct_decision_count}/{total_tests})",
        "Main_Focus_Accuracy": f"{(correct_main_focus_count / total_tests) * 100:.2f}% ({correct_main_focus_count}/{total_tests})",
        "Priority_1_Accuracy": f"{(correct_priority_1_count / total_tests) * 100:.2f}% ({correct_priority_1_count}/{total_tests})",
        "Full_Sequence_Accuracy": f"{(correct_full_sequence_count / total_tests) * 100:.2f}% ({correct_full_sequence_count}/{total_tests})",
        "Details": results
    }
    
    print("\n--- ĐÁNH GIÁ HOÀN TẤT ---")
    return summary

In [63]:
evaluate_orchestration_agent(client=client)

--- BẮT ĐẦU ĐÁNH GIÁ AGENT ĐIỀU PHỐI (18 KỊCH BẢN) ---
Đã truy xuất 5 đoạn văn bản từ kho 'blast' cho câu hỏi: 'Nguyên tắc ưu tiên điều trị bệnh blast cho lúa gia...'
Đã truy xuất 5 đoạn văn bản từ kho 'general_qa' cho câu hỏi: 'Nguyên tắc bón phân và duy trì mực nước cho lúa 45...'
[SIMULATION] treatment_agent sẽ được kích hoạt với mục đích: Phun thuốc đặc trị Beam 75WP để điều trị bệnh Đạo ôn
[SIMULATION] water_agent sẽ được kích hoạt với mục đích: Duy trì mực nước 3-5 cm để hỗ trợ lúa trổ bông
[SIMULATION] nutrient_agent sẽ được kích hoạt với mục đích: Không bón phân trong giai đoạn này do lúa đã đạt 45 ngày tuổi và cần tập trung vào việc điều trị bệnh
[1/18] S1: Đạo ôn lá nặng – giai đoạn đẻ nhánh - P1 Match: ✅
Đã truy xuất 5 đoạn văn bản từ kho 'brown_spot' cho câu hỏi: 'Nguyên tắc ưu tiên điều trị bệnh brown_spot cho lú...'
Đã truy xuất 5 đoạn văn bản từ kho 'general_qa' cho câu hỏi: 'Nguyên tắc bón phân và duy trì mực nước cho lúa 62...'
[SIMULATION] treatment_agent sẽ được kích

{'Total_Scenarios': 18,
 'Decision_Accuracy': '77.78% (14/18)',
 'Main_Focus_Accuracy': '100.00% (18/18)',
 'Priority_1_Accuracy': '77.78% (14/18)',
 'Full_Sequence_Accuracy': '88.89% (16/18)',
 'Details': [{'Scenario': 'S1: Đạo ôn lá nặng – giai đoạn đẻ nhánh',
   'Detected_Disease': 'blast',
   'Expected_Decision': True,
   'Actual_Decision': True,
   'Decision_Match': True,
   'Expected_Focus': 'TREATMENT',
   'Actual_Focus': 'TREATMENT',
   'Focus_Match': True,
   'P1_Agent_Match': True,
   'Full_Sequence_Match': True,
   'LLM_Reasoning': 'Đạo ôn đã được phát hiện với độ tin cậy 92% và đang ở giai đoạn 45 ngày sau khi gieo trồng. Điều này yêu cầu phải có hành động điều trị ngay lập tức để ngăn chặn sự lây lan của bệnh. Độ ẩm đất là 68%, không có rủi ro mưa lớn, nhưng cần phải xử lý bệnh trước khi xem xét các yếu tố khác.'},
  {'Scenario': 'S2: Đốm nâu nặng – giai đoạn làm đòng',
   'Detected_Disease': 'brown_spot',
   'Expected_Decision': True,
   'Actual_Decision': True,
   'Decis